In [1]:
import pandas as pd
import numpy as np
import time as tiime
import random
import os
import string 
import string
import urllib.request
import time
import re
import json
import requests
from tqdm import tqdm
import math

In [2]:
def get_content(url_id):
    url = "https://www.chinatax.gov.cn/queryManuscriptAssociation"
    headers = {"Accept": "*/*","Content-Type": "application/x-www-form-urlencoded; charset=UTF-8","Origin": "https://fgk.chinatax.gov.cn","Referer": "https://fgk.chinatax.gov.cn/","User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0",}
    sent_form = {"id": url_id}
    res = requests.post(url, headers=headers, data=sent_form, timeout=(10, 20))

    data = res.json()

    records = data["results"]["data"]["results"]

    def flatten_domain_meta(rec):
        """把一个记录里的 domainMetaList 拍平成 {<code>.<key>: value} 的字典"""
        out = {}
        for dm in rec.get('domainMetaList', []) or []:
            code = dm.get('domainMetadataCode') or dm.get('domainMetadataName') or 'meta'
            for item in dm.get('resultList', []) or []:
                k = item.get('key') or item.get('name')
                v = item.get('value')
                if k:  # 生成列名：如 'zhengcewenjian.writtentext'
                    out[f'{code}.{k}'] = v
        return out

    flat_rows = []
    for rec in records:
        base = {k:v for k,v in rec.items() if k not in ('domainMetaList','channel')}
        # 频道也可以合并为字符串列（如按 channelName 拼成 '政策法规;财税文件'）
        ch_names = [c.get('channelName','') for c in rec.get('channel',[]) or []]
        base['channel.joined'] = ';'.join(ch_names)
        # 合并元数据拍平结果
        base.update(flatten_domain_meta(rec))
        flat_rows.append(base)

    df_flat = pd.DataFrame(flat_rows)

    # 可选：把几个时间戳转为 datetime
    for c in ['publishedTime','modifiedTime','sortedTime','createdTime','seqNum']:
        if c in df_flat.columns:
            df_flat[c] = pd.to_datetime(df_flat[c], unit='ms', errors='coerce')
            
    return df_flat

In [3]:
tb = pd.read_pickle('./list.pkl')
tb['url_id'] = tb.url.apply(lambda x: x.split('/')[-2][1:])
tb.sample()

,title,url,publishedTimeStr,channelName,ywlj,author,ldzw,ycjr,source,lsxx,...,taxpayertype,revisesummary,aging,formulatedyear,taxdiscount,accepteddepartments,texteditionattach,writtendepartments,sftsgj,url_id
2074,国家税务总局关于下放城镇土地使用税困难减免税审批权限有关事项的公告,http://www.chinatax.gov.cn/zcfgk/c100012/c5194...,2014-01-08 00:00:00,税务规范性文件,,,,,SZfaguiku,,...,,null,已修改,2014,,财产和行为税一处,,国家税务总局,null,5194432


In [4]:
tb = tb.loc[:, ['url_id']].copy()
tb.sample()

,url_id
860,5202300


In [5]:
path_dic = './fg/'
remove_list  = []
for root, dirs, files in os.walk(path_dic):
    for name in files:
        remove_list.append(name.split('.')[0])
df_filtered = tb[~tb["url_id"].isin(remove_list)].copy()
tb_copy = df_filtered.copy()
tb_copy.reset_index(drop=True, inplace=True)
print(tb_copy.shape)

(1, 1)


In [6]:
for index, row in tqdm(tb_copy.iterrows(), total=tb_copy.shape[0]):
    url_id = row.url_id
    df_flat = get_content(url_id)
    df_flat.to_pickle('./fg/' + url_id + '.pkl')  
    time.sleep(random.uniform(10, 12))

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:11<00:00, 11.52s/it]
